# AIC2026 — SƠ TUYỂN ĐỢT 1 · 21/08/2026 · 19:30–22:30

## Đọc dòng này trước

Hệ thống chạy đủ 24 câu trong **45 giây** và **máy của bạn có gần như mọi thứ**.
Thứ duy nhất máy thiếu là **ảnh keyframe** (28,66 GiB) — chỉ cần cho 3 câu QA
dùng VLM nhìn ảnh.

Nên đường đi khuyến nghị là:

| | ở đâu | cho gì | thời gian |
|---|---|---|---|
| **1** | **máy bạn** | 21 câu KIS/TRAKE + QA từ OCR | ~1 phút/lượt |
| 2 | Kaggle | QA có VLM nhìn ảnh (nếu còn giờ) | ~15 phút |

Chạy local trước là để có **một điểm thật trong tay lúc 20:00**, không phụ thuộc
Kaggle sống hay chết. Notebook này là **đường 2**.

## Điểm đã biết, để so

| bản | public |
|---|---|
| L1 thuần máy | 5,6 / 12 |
| L5 (QA VLM + 3 ghim) | 6,0 / 12 |
| **L7 (OCR + ASR + ghim)** | **? — nộp lần cuối vòng thử nghiệm** |
| đỉnh bảng | 11,2 |

## Chuẩn bị trước 19:00 — Add Input

| dataset | để làm gì |
|---|---|
| code của bạn *(bản 21/08)* | — |
| `aic2026_data_nhe` · `aic2026_siglip2_so400m` · `aic2026_siglip2` | vector ảnh |
| `nguyenminhtric/data-batch-1-aic` | keyframe cho QA VLM |
| 🆕 `selaninguyen/11ntoe` | **OCR per-keyframe** |
| 🆕 `nguyenminhtric/20-note` | **ASR per-khoảng-shot** |

Secrets: `GEMINI_API_KEY`, `GROQ_API_KEY`, `CLOUDFLARE_API_TOKEN`,
`CLOUDFLARE_ACCOUNT_ID`. Internet ON. GPU không bắt buộc.

## Ô 1 — SDK Gemini, rồi **Restart kernel**

Kaggle cài sẵn `google-genai` < 2.0.0; với bản đó **mọi** lời gọi Gemini trả
`400 legacy Interactions API` — đọc y hệt lỗi hết quota.

In [ ]:
!pip install -q -U "google-genai>=2.3.0"
import google.genai as g
print("google-genai:", g.__version__, " (cần >= 2.3.0)")
print("\n>>> RESTART KERNEL NGAY BÂY GIỜ <<<")

## Ô 2 — Tìm code, kiểm đúng bản 21/08

In [ ]:
import os, shutil, pathlib, json, time

def thu_muc(goc, sau, bo_qua=('keyframes',)):
    muc, ra = [goc], []
    for _ in range(sau):
        tiep = []
        for d in muc:
            if os.path.basename(d) in bo_qua:
                continue
            try:
                tiep += [e.path for e in os.scandir(d) if e.is_dir()]
            except OSError:
                pass
        ra += tiep
        muc = tiep
    return ra

DIRS = thu_muc('/kaggle/input', 7)
goc = next((d for d in DIRS if os.path.exists(os.path.join(d, 'nop_bai.py'))), None)
assert goc, "Không thấy dataset code (dấu hiệu: nop_bai.py)"
shutil.copytree(goc, '/kaggle/working/sys', dirs_exist_ok=True)
os.chdir('/kaggle/working/sys')
print("code:", goc, "\n")

kiem = {
    "OCR per-keyframe  aic2026/retrieval/kho_ocr_fts.py":
        pathlib.Path('aic2026/retrieval/kho_ocr_fts.py').exists(),
    "kênh chữ khoảng   aic2026/retrieval/kenh_chu.py":
        pathlib.Path('aic2026/retrieval/kenh_chu.py').exists(),
    "lớp ghim          aic2026/core/ghim.py":
        pathlib.Path('aic2026/core/ghim.py').exists(),
    "cache LLM         out/viet_lai_truy_van.json":
        pathlib.Path('out/viet_lai_truy_van.json').exists(),
}
for k, v in kiem.items():
    print(("  ✅ " if v else "  ❌ ") + k)
assert all(kiem.values()), "Dataset code là BẢN CŨ — upload lại rồi chạy lại."

## Ô 3 — Khoá API

In [ ]:
from kaggle_secrets import UserSecretsClient
uc = UserSecretsClient()
for ten, bb in [("GEMINI_API_KEY", True), ("GROQ_API_KEY", False),
                ("CLOUDFLARE_API_TOKEN", False), ("CLOUDFLARE_ACCOUNT_ID", False)]:
    try:
        os.environ[ten] = uc.get_secret(ten)
        print(f"  ✅ {ten}: {len(os.environ[ten])} ký tự")
    except Exception:
        print(f"  {'❌' if bb else '—'} {ten}")
        if bb:
            raise

## Ô 4 — Dò dataset + dựng config

Tự tìm `ocr_fts.sqlite` (Stage11) và `shadow_evidence_pack_registry.*`
(Stage20). Thiếu cái nào thì kênh đó tự tắt, hệ thống vẫn chạy — **không mất
gì so với hôm qua**.

In [ ]:
def tim_thu_muc(ten):
    return sorted(d for d in DIRS if os.path.basename(d) == ten)

def tim_file(ten):
    for d in DIRS + ['/kaggle/input']:
        p = os.path.join(d, ten)
        if os.path.isfile(p):
            return p
    return None

nhe    = tim_thu_muc('aic2026_data_nhe')
so400m = tim_thu_muc('aic2026_siglip2_so400m')
base   = tim_thu_muc('aic2026_siglip2')
kf     = tim_thu_muc('keyframes')
assert nhe and so400m and base, "Thiếu dataset vector"

OCR = tim_file('ocr_fts.sqlite')
ASR = (tim_file('shadow_evidence_pack_registry.parquet')
       or tim_file('shadow_evidence_pack_registry.csv'))
print(f"  keyframes : {len(kf)} thư mục")
print(f"  OCR       : {OCR or '❌ thiếu 11ntoe -> kênh OCR TẮT'}")
print(f"  ASR       : {ASR or '❌ thiếu 20-note -> kênh ASR TẮT'}")

def ghi_cfg(ten, kenh=True):
    s = ("extends: config.example.yaml\n"
         "offline:\n"
         f"  btc_data_root: {nhe[0]}\n"
         f"  image_feature_root: {so400m[0]}\n"
         "retrieval:\n  hop_nhat:\n    phu:\n"
         f"    - image_feature_root: {base[0]}\n")
    muc = []
    if kenh and OCR:
        muc.append("    - ten: OCR-keyframe\n      bat: true\n"
                   "      kieu: fts\n"
                   f"      duong_dan: {OCR}\n"
                   "      bang: OCR_ALL\n      trong_so: 2.0\n      pool: 300\n")
    if kenh and ASR:
        muc.append("    - ten: ASR-khoang\n      bat: true\n"
                   "      kieu: khoang\n"
                   f"      duong_dan: {ASR}\n"
                   "      trong_so: 1.0\n      dung_ocr: false\n"
                   "      dung_asr: true\n      pool_khoang: 120\n"
                   "      toi_da_event_moi_khoang: 12\n      nguong_diem: 0.15\n")
    if muc:
        s += "  kenh_chu:\n" + "".join(muc)
    if kf:
        s += "qa:\n  image_roots:\n" + "".join(f"  - {p}\n" for p in kf)
    open(ten, 'w', encoding='utf-8').write(s)
    return ten

ghi_cfg('cfg_kenh.yaml', kenh=True)
ghi_cfg('cfg_nen.yaml',  kenh=False)
print("\n" + open('cfg_kenh.yaml', encoding='utf-8').read())

## Ô 5 — Soi kênh chữ trước khi tin *(0 request, ~1 phút)*

Kênh chữ **đã từng làm hỏng điểm một lần** (bật metadata 08/08: −0.0300 ±
0.0104). Ô này kiểm ba điều trước khi dùng: nạp được không, gắn đủ 177.321
event không, và tra thử có ra đúng thứ không.

In [ ]:
from aic2026 import config as ch
cfg = ch.load_config('cfg_kenh.yaml')
kenh = ch.nap_kenh_chu(cfg)
print(f"\n{len(kenh)} kênh chữ bật")

ev, emb, metas, rep = ch.nap_du_lieu(cfg)
print(f"{rep.events:,d} event\n")
for kho, w in kenh:
    n = kho.gan_event(ev)
    print(f"  {type(kho).__name__:14s} w={w:<4g} gắn {n:,d}/{rep.events:,d}"
          + ("  ✅" if n == rep.events else "  ❌ LỆCH — kênh này sẽ câm"))

THU = ["cá mập thị trấn ven biển du khách",
       "lễ hội điêu khắc trên cát",
       "đàn hổ con quý hiếm miền Nam"]
for q in THU:
    print(f"\n▸ {q}")
    for kho, w in kenh:
        ds = kho.xep_hang_event(q, 3)
        print(f"   {type(kho).__name__:14s} {[e for e, _ in ds][:3]}")

## Ô 6 — *(tuỳ chọn)* A/B trên dev set 120 câu

Chỉ chạy nếu còn ≥25 phút trước 19:30. Trọng số đã dò trên bộ đề thử nghiệm
(xem `config_ocr_asr.yaml`), ô này chỉ để xác nhận không có hồi quy trên dev
set — nơi truy vấn là **mô tả cảnh**, tức chỗ kênh chữ YẾU nhất.

Số nền (so400m + hợp nhất, 120 câu KIS tiếng Việt): **VI thô 0.2883 · LLM viết
lại 0.3450 · người gõ tay 0.3750**.

In [ ]:
# !python -X utf8 do_diem_thuc_te.py --config=cfg_nen.yaml  --vi 2>&1 | tail -22
# !python -X utf8 do_diem_thuc_te.py --config=cfg_kenh.yaml --vi 2>&1 | tail -22
print("bỏ comment nếu còn thời gian")

---
# ĐỀ THẬT — 19:30

## Ô 7 — Đọc đề · **ĐỌC HẾT CẢNH BÁO**

In [ ]:
DE_DIR = "/kaggle/input/de-dot1"      # sửa cho đúng
get_ipython().system(
    f'python -X utf8 doc_de_thi.py {DE_DIR} --ra data/de_dot1.json --xem 2')

⚠ Hai thứ phải soi bằng mắt — cả hai cho 0 điểm mà zip vẫn hợp lệ:

* **`task`** của từng câu;
* **số sự kiện TRAKE** — đếm tay trong đề rồi đối chiếu. Bộ thử nghiệm có
  `query-p1-16` với câu dẫn đứng trước E1..E4 (dễ đọc thành 5) và
  `query-p1-18` bị BTC đánh nhầm E1/E2/**E2**/E4.

## Ô 8 — Chạy + **nộp ngay lần 1** (~20:15)

In [ ]:
get_ipython().system(
    'python -X utf8 nop_bai.py --config cfg_kenh.yaml '
    '--truy-van data/de_dot1.json --ra /kaggle/working/dot1_L1.zip')

Trong output, tìm hai dòng này:

```
kênh chữ: KhoOcrFts    · trọng số 2.0 · 177,321 event đã gắn
kênh chữ: KhoChuKhoang · trọng số 1.0 · 177,321 event đã gắn
```

Thấy `❌ KHÔNG GẮN ĐƯỢC EVENT` thì kênh đó đang câm — bài nộp vẫn hợp lệ nhưng
bạn đang chạy như hôm qua.

## Ô 9 — Soi bài nộp + tra OCR cho câu nghi ngờ

`soi_ket_qua.py` dùng metadata (0 request). Với câu nào tiêu đề hạng 1 không ăn
nhập, tra thẳng **OCR** — hôm nay cách này đọc ra nguyên văn đáp án cho 7/24
câu của bộ thử nghiệm.

In [ ]:
get_ipython().system(
    'python -X utf8 soi_ket_qua.py /kaggle/working/dot1_L1.zip '
    '--config cfg_kenh.yaml --de data/de_dot1.json --sau 3 '
    '--ra /kaggle/working/soi_dot1.md 2>&1 | tail -45')

In [ ]:
# TRA OCR — dán truy vấn nghi ngờ vào đây. 0 request, vài giây.
import json as _j
de = {q["qid"]: q for q in _j.load(open("data/de_dot1.json", encoding="utf-8"))["queries"]}
moc, tong = [], 0
from aic2026.core import dataset as _ds
_paths = _ds.scan(cfg["offline"]["btc_data_root"])
_tieu = {}
for vid in sorted(_paths):
    vp = _paths[vid]
    if not vp.du_toi_thieu:
        continue
    rows = _ds.load_map_keyframes(vp.mapping)
    moc.append((tong, tong + len(rows), vid, rows)); tong += len(rows)
    if vp.meta is not None:
        try:
            _tieu[vid] = _ds.load_media_info(vp.meta, vid).title
        except Exception:
            pass

def tra_ocr(qid_hoac_text, k=4):
    t = de.get(qid_hoac_text, {}).get("text", qid_hoac_text)
    print(f"\n▸ {t[:88]}")
    for kho, _w in kenh:
        if type(kho).__name__ != "KhoOcrFts":
            continue
        for fr, s in kho.cham(t)[:k]:
            for a, b, vid, rows in moc:
                if a <= fr < b:
                    r = rows[fr - a]
                    print(f"   {s:.3f} {vid} kf={r.n} frame={r.frame_idx}"
                          f"  {_tieu.get(vid, '')[:44]}")
                    print(f"          {kho.van_ban(fr)[:104]}")
                    break

for _q in list(de)[:3]:
    tra_ocr(_q)

## Ô 10 — Ghim những câu OCR đọc ra bằng chứng

Chỉ ghim khi có **bằng chứng đọc được**, không ghim theo linh cảm. Mỗi mục phải
ghi lại chữ OCR đã thấy vào `ghi_chu` — để 21 giờ tối còn kiểm lại được.

In [ ]:
GHIM = r"""
{
  "_ghi_chu": "mỗi mục PHẢI kèm chữ OCR đã đọc được, không ghim theo linh cảm"
}
"""
import pathlib
d = json.loads(GHIM)
pathlib.Path('data/ghim_dot1.json').write_text(GHIM, encoding='utf-8')
n = len([k for k in d if not k.startswith('_')])
print(f"{n} truy vấn được ghim")
if n:
    get_ipython().system(
        'python -X utf8 nop_bai.py --config cfg_kenh.yaml '
        '--truy-van data/de_dot1.json --ghim data/ghim_dot1.json '
        '--ra /kaggle/working/dot1_L2.zip')

## Ô 11 — Kiểm lần cuối

**3 lần nộp mỗi gói, chỉ tính LẦN CUỐI.** Nộp sai định dạng vẫn trừ một lần.

In [ ]:
import zipfile, csv, io as _io
for z in sorted(pathlib.Path('/kaggle/working').glob('dot1_*.zip')):
    zz = zipfile.ZipFile(z); ten = zz.namelist(); xau = []
    for n in ten:
        if not n.endswith('.csv'):
            continue
        rows = list(csv.reader(_io.StringIO(zz.read(n).decode('utf-8'))))
        cot = {len(r) for r in rows}
        task = 'trake' if 'trake' in n else 'qa' if 'qa' in n else 'kis'
        if len(rows) > 100 or len(cot) != 1:
            xau.append(n)
        if task == 'kis' and cot != {2}:
            xau.append(n)
        if task == 'qa':
            if cot != {3}:
                xau.append(n)
            elif max(len(','.join(r[2:])) for r in rows) > 100:
                xau.append(n + ' (answer > 100 ký tự)')
    print(f"{z.name:20s} {len(ten):3d} file · "
          f"gốc={sorted({n.split('/')[0] for n in ten})} · "
          + ("✅ hợp lệ" if not xau else f"❌ {sorted(set(xau))[:4]}"))

---
# ⭐ VLM XẾP LẠI TOP-100 — đòn lớn nhất còn lại

Điểm đã leo **5,6 → 6,0 → 7,4 → 7,6** nhờ tìm đúng VIDEO. Phần còn thiếu là
tìm đúng **KHUNG**: KIS chỉ ăn điểm khi `frame_idx` nằm trong cửa sổ ~107 khung
(≈ 2 keyframe) của BTC.

Soi bài nộp 7,6 điểm: video hạng 1 gần như chắc đúng ở nhiều câu, nhưng chỉ có
**1–11 khung** của video đó nằm trong 100 dòng. Đó là chỗ mất điểm.

Đã thử sửa bằng cách **dồn slot** (phân bổ lại 100 dòng theo video) — đo trên
120 câu dev có ground truth cấp khung: **thất bại**, R@100 sập 45% → 29%. Cái
đuôi rải đều đang mua thứ thật.

Đường còn lại là **nhìn ảnh**. `gemini-3.5-flash-lite` chọn đúng ảnh khớp mô tả
**9/10 với mô tả tiếng Việt** trên 120 ảnh dev có GT (đoán mò 1/10). Đây là
việc mà mắt người vẫn làm ở các đội mạnh — chỉ khác là tự động.

Chi phí: 10 request/câu × 21 câu ≈ **210 request, ~20 phút**. Bậc lite RPD 500.
Cache ghi sau mỗi câu nên đứt quota vẫn đi tiếp được.

In [ ]:
get_ipython().system(
    'python -X utf8 chay_rerank.py /kaggle/working/dot1_L2.zip '
    '--config cfg_kenh.yaml --de data/de_dot1.json '
    '--sau 100 --che-do rrf --model gemini-3.5-flash-lite '
    '--ra /kaggle/working/dot1_L3rerank.zip')

### Đọc kỹ ba con số trước khi nộp

* **`hạng 1 đổi`** — 0 nghĩa là VLM không cãi encoder câu nào (đáng ngờ); gần
  hết cũng đáng ngờ. Khoảng 30–70% là hợp lý.
* **`lô hoà điểm`** — lô mà mọi ảnh cùng một điểm = model không nhìn ảnh. Đúng
  cái bẫy đã bắt được ở `gemini-3.6-flash` ngày 20/08.
* **đổi KHUNG trong cùng video** ăn điểm chắc hơn **đổi hẳn VIDEO**.

In [ ]:
import zipfile, csv, io as _io
cu  = zipfile.ZipFile('/kaggle/working/dot1_L2.zip')
moi = zipfile.ZipFile('/kaggle/working/dot1_L3rerank.zip')

def dau(z, n):
    r = list(csv.reader(_io.StringIO(z.read(n).decode('utf-8'))))
    return r[0][0], r[0][1]

doi_khung = doi_video = giu = 0
for n in sorted(cu.namelist()):
    if 'trake' in n:
        continue
    (v1, f1), (v2, f2) = dau(cu, n), dau(moi, n)
    if (v1, f1) == (v2, f2):
        giu += 1
    elif v1 == v2:
        doi_khung += 1
        print(f"  {n[11:-4]:22s} cùng video {v1}: khung {f1} -> {f2}")
    else:
        doi_video += 1
        print(f"  {n[11:-4]:22s} ĐỔI VIDEO {v1}/{f1} -> {v2}/{f2}")
print(f"
  giữ nguyên {giu} · đổi KHUNG {doi_khung} · đổi VIDEO {doi_video}")